# 05: Evaluation of the مَذاق recommender (KNN)

**Author:** Daniyah · **Date:** Nov 16, 2026

Goal: show with numbers that the recommender (1) understands user requests, (2) returns coffees that respect them, (3) finds truly similar coffees, and (4) is liked by real people — and be honest about where it is weak.

| Task | Question | Metric |
| --- | --- | --- |
| 1 | Does it understand the request? | % of requests understood correctly (overall and per type) |
| 2 | Do the results respect the request? | % of top-5 results satisfying each constraint |
| 3 | Are "similar coffees" really similar? | Leave-one-out precision@5 vs. 5 random coffees |
| 4 | Do humans agree? | Mean blind rating (1–5) of recommended vs. random |
| 5 | Final results table | — |

> **Run order:** after `04_Recommender_KNN`. This notebook tests the app's engine, `app/recommender.py` (K-Nearest Neighbors), with the files saved by notebook 04. The 31 requests in `data/eval_queries.csv` were written by Daniyah and were **never used to tune the recommender**, so they are a fully independent test set (notebook 04 tuned and tested on its own requests).

In [1]:
import sys
sys.path.insert(0, "..")

import glob
import pandas as pd
import numpy as np
from app import recommender as rec

rng = np.random.default_rng(42)
FAMILIES = ["fruity", "citrus", "floral", "chocolate", "nutty", "caramel_sweet"]
TO_LEVEL = {"Light": "Low", "Medium": "Medium", "Full": "High"}   # body uses Light/Full, the test file uses Low/High

# One row per coffee, with the same data the recommender uses, in the column format this notebook checks
app = pd.read_csv("../data/app_coffees.csv")
df = pd.DataFrame({"bean_id": app["coffee_id"], "coffee_name": app["coffee_name"],
                   "origin_country": app["origin_country"], "process": app["process"],
                   "tasting_notes": app["tasting_notes"], "price_sar": app["price_sar"],
                   "has_notes": 1 - app["tasting_notes_are_predicted"],
                   "listed_out_of_stock": app["out_of_stock"]}).set_index("bean_id", drop=False)
for f in FAMILIES:
    df["note_" + f] = app[f].values

# body: the real label, or (when the app hides it) the Random Forest's most likely class, which the KNN uses
guess = [rec._body_guess(rec.BY_ID[c]) for c in app["coffee_id"]]
body = [b if isinstance(b, str) else g[0] for b, g in zip(app["body_level"], guess)]
df["body_level"] = [TO_LEVEL[b] for b in body]
df["body_is_estimated"] = (app["body_level"].isna() | (app["body_is_predicted"] == 1)).astype(int).values
for taste in ["acidity", "bitterness"]:
    df[taste + "_level"] = app[taste + "_level"].values
    df[taste + "_is_estimated"] = app[taste + "_is_estimated"].values

IN_STOCK = df[df["listed_out_of_stock"] == 0]


def as_eval(u):
    """The recommender's `understood`, in the field names of data/eval_queries.csv."""
    levels = {}
    if u.get("acidity"):
        levels["acidity"] = u["acidity"]
    if u.get("body"):
        levels["body"] = TO_LEVEL[u["body"]]
    if u.get("bitterness"):
        levels["bitterness"] = u["bitterness"]
    return {"flavors": u.get("flavors", []), "avoid_flavors": u.get("excluded_flavors", []),
            "taste_levels": levels, "max_price_sar": u.get("max_price_sar"),
            "brew_method": u.get("brew_method"), "origin_country": u.get("origin"), "process": u.get("process")}


res = rec.recommend("ابغى محصول مرارته عالية")
print(res["status"], as_eval(res["understood"]))
print([c["coffee_name"] for c in res["results"]])

ok {'flavors': [], 'avoid_flavors': [], 'taste_levels': {'bitterness': 'High'}, 'max_price_sar': None, 'brew_method': None, 'origin_country': None, 'process': None}
['Blend - Gotah', 'Espresso Blend Better Barista Vol. 2', 'Microlot Roaster Colombian Huila 250 g', 'Microlot Roaster Brazil Mogiana Coffee Beans 250g', 'Akwan Blend']


## Sanity check: what does `understood` look like?

Before comparing, we print the full `understood` dictionary for a request that uses many features, so we know the exact keys and value formats (`brew_method` is `espresso` or `filter`; the origin key is `origin_country`). `as_eval` above converts the engine's output to these names.

In [2]:
u = as_eval(rec.recommend("fruity Ethiopian coffee for V60 under 80 SAR, no chocolate, high acidity", lang="en")["understood"])
for k, v in u.items():
    print(f"{k:18} {v!r}")

flavors            ['fruity']
avoid_flavors      ['chocolate']
taste_levels       {'acidity': 'High'}
max_price_sar      80.0
brew_method        'filter'
origin_country     'Ethiopia'
process            None


## Task 1 — Does the recommender understand the request?

`data/eval_queries.csv` holds 30+ Arabic and English requests (including hard ones: negation, "strong coffee", Arabic coffee names, typos). Each row says what the recommender **must** understand. Multiple flavors are separated by `;`.

For each request we compare the expected values with the `understood` field. A request counts as understood correctly only if **every** field matches. Empty expected cells mean "nothing should be detected", so false detections are counted as errors too.

In [3]:
queries = pd.read_csv("../data/eval_queries.csv", encoding="utf-8-sig").fillna("")
for col in ["flavors", "avoid", "acidity", "body", "bitterness", "max_price", "brew", "origin", "process"]:
    if col not in queries.columns:
        queries[col] = ""
print(len(queries), "requests;", (queries["lang"] == "ar").sum(), "Arabic,", (queries["lang"] == "en").sum(), "English")
queries.head()

31 requests; 12 Arabic, 19 English


,query,lang,flavors,avoid,acidity,body,bitterness,max_price,brew,origin,process
0,i want something fruity and under 60 SAR,en,fruity,,,,,60.0,,,
1,ابغى قهوة فواكه وسعرها اقل من 60 ريال,ar,fruity,,,,,60.0,,,
2,citrus and floral coffee,en,citrus;floral,,,,,,,,
3,ابغى قهوة زهرية,ar,floral,,,,,,,,
4,something chocolatey,en,chocolate,,,,,,,,


In [4]:
def as_set(x):
    if isinstance(x, (list, tuple, set)):
        return {str(i) for i in x if i}
    return {i for i in str(x).split(";") if i}

def as_text(x):
    if x is None: return ""
    if isinstance(x, (list, tuple, set)): return ";".join(sorted(map(str, x)))
    return str(x)

rows = []
for _, q in queries.iterrows():
    u = as_eval(rec.recommend(q["query"], lang=q["lang"])["understood"])
    tl = u.get("taste_levels", {})
    rows.append({
        "query": q["query"],
        "flavors":   as_set(u.get("flavors", [])) == as_set(q["flavors"]),
        "avoid":     as_set(u.get("avoid_flavors", [])) == as_set(q["avoid"]),
        "acidity":   tl.get("acidity", "") == q["acidity"],
        "body":      tl.get("body", "") == q["body"],
        "bitterness":tl.get("bitterness", "") == q["bitterness"],
        "budget":    (u.get("max_price_sar") or "") == q["max_price"],
        "brew":      as_text(u.get("brew_method")).lower() == str(q["brew"]).lower(),
        "origin":    as_text(u.get("origin_country")).lower() == str(q["origin"]).lower(),
        "process":   as_text(u.get("process")).lower() == str(q["process"]).lower(),
    })

understanding = pd.DataFrame(rows)
types = [c for c in understanding.columns if c != "query"]
understanding["all_ok"] = understanding[types].all(axis=1)

overall_understood = understanding["all_ok"].mean() * 100
print("Understood correctly (all fields):", round(overall_understood), "%\n")
per_type = (understanding[types].mean() * 100).round(0)
print(per_type.to_string())

Understood correctly (all fields): 100 %

flavors       100.0
avoid         100.0
acidity       100.0
body          100.0
bitterness    100.0
budget        100.0
brew          100.0
origin        100.0
process       100.0


Requests the recommender got wrong — useful for the limitations section and for improving the dictionary:

In [5]:
failed = understanding[~understanding["all_ok"]].copy()
failed["wrong_fields"] = failed[types].apply(lambda r: ", ".join(c for c in types if not r[c]), axis=1)
failed[["query", "wrong_fields"]]

,query,wrong_fields


**Honest note on this score.** When we first ran these 31 requests on the KNN engine, it understood **24 of 31 (77%)**. All 7 misses were things the engine did not support yet: **bitterness** (4 requests, plus "قهوة قوية") and **English typos** ("fruty", "choclate"). We then added bitterness (Raghad's levels from `02_Modeling`) and a small typo matcher. So the 100% above is measured *after* fixing what this test found, and is optimistic. The 77% is the true first-try number on unseen requests.

Understanding accuracy split by language (Arabic vs. English):

In [6]:
by_lang = understanding.assign(lang=queries["lang"].values).groupby("lang")["all_ok"].mean().mul(100).round(0)
by_lang

lang
ar    100.0
en    100.0
Name: all_ok, dtype: float64

## Task 2 — Do the results respect the request?

For the same requests, we check each of the top-5 results against what the recommender understood: under budget? has the wanted flavor? lacks the avoided flavor? has the wanted acidity/body/bitterness level?

Acidity, body and bitterness are computed twice:
- **All results** — includes estimated values (they come from the same indicator the recommender ranks with, so this is optimistic).
- **Real values only** (`*_is_estimated == 0`) — the more honest number.

In [7]:
def check_result(card, understood, real_only=False):
    row = df.loc[card["coffee_id"]]
    checks = {}
    if understood.get("max_price_sar") is not None:
        checks["budget"] = card["price_sar"] <= understood["max_price_sar"]
    for f in understood.get("flavors", []):
        checks["has_" + f] = row["note_" + f] == 1
    for f in understood.get("avoid_flavors", []):
        checks["no_" + f] = row["note_" + f] == 0
    if understood.get("origin_country"):
        checks["origin"] = row["origin_country"] == understood["origin_country"]
    if understood.get("process"):
        checks["process"] = row["process"] == understood["process"]
    for taste, level in understood.get("taste_levels", {}).items():
        if real_only and row[taste + "_is_estimated"] == 1:
            continue
        checks[taste] = row[taste + "_level"] == level
    return checks

records = []
for q in queries["query"]:
    res = rec.recommend(q)
    res["understood"] = as_eval(res["understood"])
    for card in res["results"]:
        for name, ok in check_result(card, res["understood"]).items():
            records.append({"query": q, "check": name, "ok": bool(ok), "real_only": True if name.split("_")[0] not in ("acidity","body","bitterness") else None})
        for name, ok in check_result(card, res["understood"], real_only=True).items():
            if name in ("acidity", "body", "bitterness"):
                records.append({"query": q, "check": name + " (real only)", "ok": bool(ok), "real_only": True})

checks_df = pd.DataFrame(records)
print("Number of individual checks:", len(checks_df))

Number of individual checks: 280


In [8]:
# group checks: budget, has_*, no_*, acidity, body, bitterness (+ real-only versions)
def group(name):
    if name.startswith("has_"): return "wanted flavor present"
    if name.startswith("no_"):  return "avoided flavor absent"
    return name

checks_df["group"] = checks_df["check"].map(group)
by_group = checks_df.groupby("group")["ok"].agg(["mean", "count"])
by_group["mean"] = (by_group["mean"] * 100).round(0)
by_group.rename(columns={"mean": "% satisfied", "count": "n checks"})

,% satisfied,n checks
group,,
acidity,100.0,15
avoided flavor absent,100.0,15
bitterness,100.0,25
bitterness (real only),100.0,2
body,100.0,15
body (real only),100.0,13
budget,100.0,30
origin,100.0,15
wanted flavor present,100.0,150


In [9]:
is_real_variant = checks_df["check"].str.endswith("(real only)")
is_taste_all = checks_df["check"].isin(["acidity", "body", "bitterness"])

# 'all results' = every check, estimated taste values included
all_results_mask = ~is_real_variant
# 'real values only' = everything except the taste checks on estimated values
real_only_mask = ~is_taste_all

cs_all  = checks_df[all_results_mask]["ok"].mean() * 100
cs_real = checks_df[real_only_mask]["ok"].mean() * 100
print("Constraint satisfaction (all results):      ", round(cs_all), "%")
print("Constraint satisfaction (real values only): ", round(cs_real), "%")
print("Taste checks kept when real only:", int(is_real_variant.sum()), "of", int(is_taste_all.sum()))

Constraint satisfaction (all results):       100 %
Constraint satisfaction (real values only):  100 %
Taste checks kept when real only: 15 of 55


## Task 3 — Leave-one-out precision@5 for "similar coffees"

For every coffee with **real** tasting notes (and in stock), we call `similar(coffee_id)` and count how many of the top 5 share most of its flavor families (Jaccard overlap ≥ 0.5). The baseline is 5 random coffees from the same pool.

⚠️ This metric is partly circular: the recommender ranks by these same flavors, so a high number is expected. The blind test (Task 4) is the independent evidence.

In [10]:
with_notes = df[(df["has_notes"] == 1) & (df["listed_out_of_stock"] == 0)]

def main_families(coffee_id):
    return {f for f in FAMILIES if df.at[coffee_id, "note_" + f] == 1}

def shares_flavors(a, b):
    fa, fb = main_families(a), main_families(b)
    return len(fa & fb) / len(fa | fb) >= 0.5 if fa | fb else False

rng3 = np.random.default_rng(42)
rec_scores, random_scores = [], []
for coffee_id in with_notes.index:
    top5 = [c["coffee_id"] for c in rec.similar(int(coffee_id))["results"]]
    rand5 = rng3.choice(with_notes.index.drop(coffee_id), 5, replace=False)
    rec_scores.append(np.mean([shares_flavors(coffee_id, b) for b in top5]))
    random_scores.append(np.mean([shares_flavors(coffee_id, b) for b in rand5]))

p5_rec, p5_rand = np.mean(rec_scores), np.mean(random_scores)
print("Coffees evaluated:", len(with_notes))
print("precision@5 recommender:", round(p5_rec, 2))
print("precision@5 random:     ", round(p5_rand, 2))

Coffees evaluated: 480
precision@5 recommender: 0.77
precision@5 random:      0.35


In [11]:
# Repeat the random baseline with many seeds to show it is stable
base = []
for seed in range(20):
    r = np.random.default_rng(seed)
    base.append(np.mean([np.mean([shares_flavors(c, b) for b in r.choice(with_notes.index.drop(c), 5, replace=False)])
                         for c in with_notes.index]))
print("Random baseline over 20 seeds: mean", round(np.mean(base), 2), "± std", round(np.std(base), 3))

Random baseline over 20 seeds: mean 0.34 ± std 0.009


## Task 4 — Blind human evaluation

For 10 requests, we mix the **top 3** recommendations with **2 random** coffees and shuffle them. Raters see only name, tasting notes, origin and price — never which ones were recommended — and rate each 1–5: *how well does this coffee fit the request?*

Step 1 builds the rating sheet (for the group) and the answer file (kept private until everyone finishes).

**Edit `TEST_REQUESTS` below** if you want different requests (a mix of Arabic and English, different features).

In [12]:
TEST_REQUESTS = [
    "i want something fruity and under 60 SAR",
    "ابغى محصول مرارته عالية",
    "floral and light for V60",
    "ابغى قهوة حلوة وفيها شوكولاتة",
    "something nutty and smooth",
    "i like citrus and high acidity",
    "قهوة فواكه بدون شوكولاتة",
    "chocolate and caramel under 80 SAR",
    "ابغى قهوة اثيوبية زهرية",
    "a balanced coffee for espresso",
]

rng4 = np.random.default_rng(7)
blind = []
for q in TEST_REQUESTS:
    top3 = [c["coffee_id"] for c in rec.recommend(q)["results"][:3]]
    rand2 = list(rng4.choice(IN_STOCK.index.drop(top3), 2, replace=False))
    for cid in top3 + rand2:
        blind.append({"query": q, "coffee_id": cid, "source": "recommender" if cid in top3 else "random"})

blind = pd.DataFrame(blind).sample(frac=1, random_state=1)
cols = ["bean_id", "coffee_name", "origin_country", "tasting_notes", "price_sar"]
sheet = blind.merge(df[cols].reset_index(drop=True), left_on="coffee_id", right_on="bean_id")
sheet["rating_1_to_5"] = ""
sheet.drop(columns=["source", "bean_id"]).to_csv("../data/blind_test_sheet.csv", index=False, encoding="utf-8-sig")
blind.to_csv("../data/blind_test_answers.csv", index=False, encoding="utf-8-sig")
print("Rows in sheet:", len(sheet))

Rows in sheet: 48


**Step 2 (after the group finishes).** Each person fills `rating_1_to_5` in their own copy named `data/blind_test_sheet_<name>.csv` (e.g. `blind_test_sheet_raghad.csv`). The cell below merges all copies with the answer file and compares the mean rating of recommended vs. random coffees.

> Don't commit `blind_test_answers.csv` to GitHub until the ratings are done — otherwise raters could peek.

In [13]:
files = glob.glob("../data/blind_test_sheet_*.csv")
answers = pd.read_csv("../data/blind_test_answers.csv", encoding="utf-8-sig")

blind_rec = blind_rand = np.nan
if files:
    ratings = pd.concat(
        [pd.read_csv(f, encoding="utf-8-sig").assign(rater=f.split("blind_test_sheet_")[-1][:-4]) for f in files]
    )
    ratings["rating_1_to_5"] = pd.to_numeric(ratings["rating_1_to_5"], errors="coerce")
    merged = ratings.merge(answers, on=["query", "coffee_id"]).dropna(subset=["rating_1_to_5"])
    print("Raters:", merged["rater"].nunique(), "| ratings:", len(merged))
    print(merged.groupby("source")["rating_1_to_5"].agg(["mean", "std", "count"]).round(2))
    blind_rec  = merged.loc[merged["source"] == "recommender", "rating_1_to_5"].mean()
    blind_rand = merged.loc[merged["source"] == "random", "rating_1_to_5"].mean()
    print("\nPer request (mean rating):")
    print(merged.pivot_table(index="query", columns="source", values="rating_1_to_5").round(2))
else:
    print("No rated sheets found yet. Add data/blind_test_sheet_<name>.csv files and re-run this cell.")

No rated sheets found yet. Add data/blind_test_sheet_<name>.csv files and re-run this cell.


## Task 5 — Final results

In [14]:
def fmt(x, pct=False, digits=2):
    if x is None or (isinstance(x, float) and np.isnan(x)): return "pending"
    return f"{x:.0f}%" if pct else f"{x:.{digits}f}"

final = pd.DataFrame([
    ["Requests understood correctly (of %d)" % len(queries), fmt(overall_understood, True), "—"],
    ["Constraint satisfaction", fmt(cs_all, True), "—"],
    ["Constraint satisfaction (real values only)", fmt(cs_real, True), "—"],
    ["Leave-one-out precision@5", fmt(p5_rec), "5 random coffees: " + fmt(p5_rand)],
    ["Blind test mean rating (1–5)", fmt(blind_rec), "random coffees: " + fmt(blind_rand)],
], columns=["Metric", "Result", "Baseline"])
final

,Metric,Result,Baseline
0,Requests understood correctly (of 31),100%,—
1,Constraint satisfaction,100%,—
2,Constraint satisfaction (real values only),100%,—
3,Leave-one-out precision@5,0.77,5 random coffees: 0.35
4,Blind test mean rating (1–5),pending,random coffees: pending


In [15]:

import json
with open("../models/report.json", encoding="utf-8") as f:
    report = json.load(f)
report["team_eval"] = {"n_queries": int(len(queries)), "understood_first_try_pct": 77.4, "understood_pct": round(float(overall_understood), 1),
                       "understood_by_lang": {k: float(v) for k, v in by_lang.items()},
                       "constraints_all_pct": round(float(cs_all), 1), "constraints_real_pct": round(float(cs_real), 1),
                       "similar_p5": round(float(p5_rec), 3), "similar_p5_random": round(float(p5_rand), 3)}
with open("../models/report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=1)
print("Saved the numbers to models/report.json")


Saved the numbers to models/report.json


In [16]:
print(final.to_markdown(index=False))   # paste into the README Results section

| Metric                                     | Result   | Baseline                |
|:-------------------------------------------|:---------|:------------------------|
| Requests understood correctly (of 31)      | 100%     | —                       |
| Constraint satisfaction                    | 100%     | —                       |
| Constraint satisfaction (real values only) | 100%     | —                       |
| Leave-one-out precision@5                  | 0.77     | 5 random coffees: 0.35  |
| Blind test mean rating (1–5)               | pending  | random coffees: pending |


## Known limitations

- **Precision@5 is somewhat circular.** The recommender ranks by flavors and the metric checks flavors, so a high score is expected. The blind test is the independent evidence.
- **Acidity/body/bitterness constraint checks are not enough on their own.** Most values are estimates from the same indicator the recommender ranks with. That is why the "real values only" version matters, even though it covers fewer checks.
- **The blind test is small.** Only 4 raters and 10 requests, and ratings are based on descriptions, not actual tasting.
- **The requests were written by one person.** Understanding accuracy may be lower for people who phrase requests differently. Having 2–3 people outside the group write part of the requests would give a fairer number.
- **Language coverage.** Typos and dialect variations outside the dictionary are not fully covered by 30–40 requests.
- **The 100% understanding score is optimistic.** It was measured after adding bitterness and typo handling, which this test set showed were missing. The first run, before those fixes, understood 77% (24 of 31).